In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import numpy as np
import torch
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV
import joblib
from tqdm.notebook import tqdm
import warnings
import json
import joblib
from sklearn.metrics import precision_score, recall_score, f1_score
warnings.filterwarnings('ignore')
# Cấu hình thiết bị 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#label
DATASET_ROOT = '/kaggle/input/vietnamese-foods'
IMG_ROOT_DIR = os.path.join(DATASET_ROOT, 'Images')

TRAIN_DIR = os.path.join(IMG_ROOT_DIR, 'Train')
VAL_DIR = os.path.join(IMG_ROOT_DIR, 'Validate')
TEST_DIR = os.path.join(IMG_ROOT_DIR, 'Test')

# Class Dataset & Transforms
# DINOv2 yêu cầu ảnh 224x224 và chuẩn hóa ImageNet
dinov2_transforms = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class FoodDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.filepaths = []
        self.labels = []
        
        if os.path.exists(root_dir):
            classes = sorted(os.listdir(root_dir))
            for label in classes:
                class_dir = os.path.join(root_dir, label)
                if os.path.isdir(class_dir):
                    for file in os.listdir(class_dir):
                        if file.lower().endswith(('.jpg', '.jpeg')):
                            self.filepaths.append(os.path.join(class_dir, file))
                            self.labels.append(label)
        
    def __len__(self):
        return len(self.filepaths)
    
    def __getitem__(self, idx):
        img_path = self.filepaths[idx]
        try:
            image = Image.open(img_path).convert("RGB")
            
            if self.transform:
                image = self.transform(image)
            return image, self.labels[idx]
        except Exception:
            return torch.zeros((3, 224, 224)), self.labels[idx]

# Khởi tạo DataLoader
print("Đang load danh sách ảnh...")
train_ds = FoodDataset(TRAIN_DIR, transform=dinov2_transforms)
val_ds = FoodDataset(VAL_DIR, transform=dinov2_transforms)
test_ds = FoodDataset(TEST_DIR, transform=dinov2_transforms)

# Gộp Train + Val để huấn luyện tốt hơn
full_train_ds = torch.utils.data.ConcatDataset([train_ds, val_ds])

BATCH_SIZE = 64
train_loader = DataLoader(full_train_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Tổng ảnh Train + Validate: {len(full_train_ds)}")
print(f"Tổng ảnh Test: {len(test_ds)}")

In [ ]:
# Tải Model DINOv2 (Small)
print("Đang tải model DINOv2 (ViT-S/14)...")
model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
model.to(device)
model.eval() # Chế độ đánh giá (không train model này)

# 2. Hàm trích xuất
def extract_features(dataloader, model_name="Dữ liệu"):
    features_list = []
    labels_list = []
    
    print(f"Bắt đầu trích xuất đặc trưng cho {model_name}...")
    with torch.no_grad(): 
        for images, labels in tqdm(dataloader):
            images = images.to(device)
            
            # Forward pass
            outputs = model(images)
            
            # Lưu kết quả
            features_list.append(outputs.cpu().numpy())
            labels_list.extend(labels)
            
    # Nối các batch lại thành ma trận lớn
    features = np.concatenate(features_list, axis=0)
    return features, np.array(labels_list)

# Thực hiện trích xuất (Lưu thẳng vào biến toàn cục)
X_train_feats, y_train_raw = extract_features(train_loader, "Tập Train")
X_test_feats, y_test_raw = extract_features(test_loader, "Tập Test")

print("\nKích thước dữ liệu sau khi qua DINOv2:")
print(f"Train Features: {X_train_feats.shape}") 
print(f"Test Features:  {X_test_feats.shape}")

In [ ]:
le = LabelEncoder()
    
# Fit trên toàn bộ nhãn
all_labels = np.concatenate([y_train_raw, y_test_raw])
le.fit(all_labels)

y_train_enc = le.transform(y_train_raw)
y_test_enc = le.transform(y_test_raw)

# --- TẠO & LƯU LABEL MAPPING ---
label_mapping = {label: int(index) for index, label in enumerate(le.classes_)}
with open('label_mapping.json', 'w', encoding='utf-8') as f:
    json.dump(label_mapping, f, ensure_ascii=False, indent=4)
print("Đã lưu file mapping: label_mapping.json")


# 2. Cấu hình GridSearch cho Random Forest
# Khởi tạo model với tham số bạn yêu cầu
rf_model = RandomForestClassifier(
    n_estimators=500,       # Tham số cố định
    max_depth=40,           # Tham số cố định
    min_samples_split=2,    # Tham số cố định
    min_samples_leaf=1,     # Tham số cố định
    random_state=42, 
    n_jobs=-1               # Dùng hết CPU core
)
# Huấn luyện
rf_model.fit(X_train_feats, y_train_enc)
print("Huấn luyện hoàn tất.")
# --- 3. DỰ ĐOÁN VÀ ĐÁNH GIÁ ---
print("Đang dự đoán trên tập Test...")
y_pred = rf_model.predict(X_test_feats)

# Chuyển ngược lại tên món ăn
y_test_names = le.inverse_transform(y_test_enc)
y_pred_names = le.inverse_transform(y_pred)

# Tính toán các chỉ số
acc = accuracy_score(y_test_names, y_pred_names)
prec = precision_score(y_test_names, y_pred_names, average='weighted')
rec = recall_score(y_test_names, y_pred_names, average='weighted')
f1 = f1_score(y_test_names, y_pred_names, average='weighted')

print("\n--- KẾT QUẢ ĐÁNH GIÁ TỔNG HỢP ---")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-Score : {f1:.4f}")

print("\n--- CHI TIẾT TỪNG LỚP (CLASSIFICATION REPORT) ---")
print(classification_report(y_test_names, y_pred_names))



# --- 5. LƯU MODEL ---
filename = 'dinov2_rf_best_fixed.pkl'
joblib.dump(rf_model, filename)
print(f"Đã lưu model thành công tại: {filename}")

In [ ]:
# --- 4. VẼ MA TRẬN NHẦM LẪN (CONFUSION MATRIX) DẠNG PHẦN TRĂM ---
import matplotlib.pyplot as plt
import seaborn as sns

# Tính ma trận nhầm lẫn
cm = confusion_matrix(y_test_names, y_pred_names, labels=le.classes_)

# Chuyển đổi sang phần trăm (Normalize theo hàng - True Labels)
cm_percent = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-9)

plt.figure(figsize=(30, 30))
# Dùng cmap Greens cho Random Forest (giống file hog-rf trước đó để đồng bộ)
plt.imshow(cm_percent, interpolation='nearest', cmap=plt.cm.Greens) 
plt.title("Confusion Matrix (%) - DINOv2 + Random Forest")
plt.colorbar()

tick_marks = np.arange(len(le.classes_))
plt.xticks(tick_marks, le.classes_, rotation=90, fontsize=12)
plt.yticks(tick_marks, le.classes_, fontsize=12)

# Hiển thị số phần trăm trong từng ô
thresh = cm_percent.max() / 2
for i in range(cm_percent.shape[0]):
    for j in range(cm_percent.shape[1]):
        # Chỉ hiển thị text nếu giá trị > 0.1% để biểu đồ thoáng
        if cm_percent[i, j] > 0.001:
            plt.text(j, i, f"{cm_percent[i, j]*100:.1f}%",
                     horizontalalignment="center",
                     verticalalignment="center",
                     color="white" if cm_percent[i, j] > thresh else "black",
                     fontsize=10)

plt.xlabel("Predicted Label", fontsize=14)
plt.ylabel("True Label", fontsize=14)
plt.tight_layout()
plt.show()